# Optimization

## I. Host and device

Minimize this, cause they are slow

### a. Pinned mem

Reduction in using pinned vs pageable

```-query-gpu=pcie.gen.current,pcie.link.gen.max```

- PCIe link is often the bottleneck for data transfers between CPU memory and GPU memory

- pageable <<< pinned -> transfer size vs bandwidth

- One big transfer better than many small -> up to some point

In [ ]:
!nvfortran opt/1_al.cuf
!./a.out

## II. Device mem

- Global -> device atr
- Local, regs -> on-chip regs, per-thread access
- Shared mem -> inside thread block
- Constant -> with constant atr

![Mem](imgs/1.png)

#### a. Global mem

- Assumed sizes: a(nx, ny, *)

- Assumed shapes: a(:, :, :) -> from array passed : use more registers (in every thread)

##### Thread warps

All data transactions are 32, 64 or 128 byte mem -> contiguos

- Offsets dont change that -> still contiguos : offten add 1 (2:1)

- Strides do change that -> not contiguos : offten add 32 (32:1)

- Cache line is 128 bytes

- All threads in warp execute at the same time -> mem trans per warp

- The offset may cross the 128 byte boundary

##### Access flow

Thread -> L1 -> L2 ->  DRAM

If not in L1, warps need to switch from L1 and bring from DRAM

In [ ]:
#offset
!nvfortran opt/2_glob.cuf
!./a.out

In [ ]:
#stride
!nvfortran opt/3_str.cuf
!./a.out

#### b. Local mem